<a href="https://colab.research.google.com/github/hanidew/G12_IR_Evaluation/blob/precision-calculation/IRWS_Precision_Calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 2: Precision Calculation

## 2.1 Initialize the Environment and Load Data

*   Load the cleaned system runs and qrels
*   Print both datasets to ensure the data is loaded correctly





In [5]:
# import the Pandas library for handling and analyzing data
import pandas as pd

# Load the cleaned system runs ('clean_system_runs_v1.csv' file)
# store it in dataframe df_runs
df_runs = pd.read_csv('clean_system_runs_v1.csv')

# Qrels format is usually: TopicID Iteration DocID Relevance
# create a list of column names
# These names describe what each column in the Qrels file represents
qrels_cols = ['topic_id', 'iteration', 'doc_id', 'relevance']
# Load the Qrels file (qrels.trec8.adhoc' file)
# sep='\s+' : the file is not comma-separated, it is separated by spaces/tabs
# names=qrels_cols : assigns the column names
df_qrels = pd.read_csv('qrels.trec8.adhoc', sep='\s+', names=qrels_cols)

# Preview the data to ensure the data is loaded correctly
print("System Runs:")
print(df_runs.head(15))
print("\nQrels:")
print(df_qrels.head(15))

<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_411/3985517741.py:15: SyntaxWarning: invalid escape sequence '\s'
  df_qrels = pd.read_csv('qrels.trec8.adhoc', sep='\s+', names=qrels_cols)


System Runs:
    Topic_ID Iteration       Doc_ID  Rank     Score System_Name
0        401        Q0   FBIS4-9582     0  1.000000      INQ602
1        401        Q0  FBIS4-31704     1  0.843713      INQ602
2        401        Q0   FBIS4-9504     2  0.765229      INQ602
3        401        Q0  FT932-10861     3  0.756782      INQ602
4        401        Q0  FBIS3-59436     4  0.721328      INQ602
5        401        Q0  FT934-15047     5  0.715240      INQ602
6        401        Q0  FBIS3-49468     6  0.701966      INQ602
7        401        Q0   FBIS4-9157     7  0.700050      INQ602
8        401        Q0   FBIS4-9438     8  0.700050      INQ602
9        401        Q0   FBIS3-4201    10  0.686036      INQ602
10       401        Q0  FBIS4-18182     9  0.686036      INQ602
11       401        Q0   FT924-7265    11  0.677531      INQ602
12       401        Q0  FBIS3-31659    12  0.655965      INQ602
13       401        Q0  FT943-15609    13  0.648601      INQ602
14       401        Q0   FB

## 2.2 Define the Precision@k Function


*   Take top K retrieved documents
*   Get all relevant documents
*   Find the intersection (overlap)
*   Divide by K





In [7]:
# Define a function named calculate_precision_at_k
# the function takes 3 inputs
# system_results : the retrieved documents (ranking output)
# qrels : ground truth relevance judgments
# k : how many top results you want to evaluate
# basically, this function calculates: “Out of the top K results, how many are actually relevant?”
def calculate_precision_at_k(system_results, qrels, k):

    # Ensure correct ranking order
    system_results = system_results.sort_values(by='Rank')

    # 1. Get Top k Documents
    # system_results.head(k) : Takes the first K rows (top-ranked documents)
    # ['Doc_ID'] : Selects only the Doc_ID column
    # .tolist() : Converts it into a list
    top_k_docs = system_results.head(k)['Doc_ID'].tolist()

    # 2. Get Relevant Documents from Qrels
    # qrels[qrels['relevance'] > 0] : keep only the relevant documents (relevance>0)
    # ['doc_id'] : extract document ID
    # .tolist() : convert to list
    # set() : convert to set for faster lookup and remove duplicates automatically
    relevant_docs = set(qrels[qrels['relevance'] > 0]['doc_id'].tolist())

    # 3. Count Relevant Retrieved Documents
    # set(top_k_docs) : convert top_k_docs to set
    # .intersection(relevant_docs) : find focuments that are retrieved (top K) AND relevant
    # len() : count the number to find the relevant documents in top K
    num_relevant_retrieved = len(set(top_k_docs).intersection(relevant_docs))

    # 4. Apply Precision@K Formula
    return num_relevant_retrieved / k


## 2.3 Generate the Score Matrices
*   Prepare data including systems, topics, and depths
*   For each topic, filter qrels
*   For each system, filter results
*   For each depth, compute the Precision@K
*   LStore everything in matrix form

In [12]:
# 1. Get all systems and topics
systems = df_runs['System_Name'].unique() # get all unique retrieval systems
topics = sorted(df_runs['Topic_ID'].unique()) # get all query topics and sort in ascending order

# 2. Define evaluation depths
# 5-very top results, 10-standard evaluation, 20-medium depth, 50-deep ranking, 100-very deep retrieval
depths = [5, 10, 20, 50, 100]

# 3. Prepare storage structure
results = {k: [] for k in depths} # each list will store rows of results for each topic

# 4. Loop trhough each topic
for topic in topics:
    # create structure for each topic
    topic_results = {k: {'Topic': topic} for k in depths}
    # get qrels for the topic
    current_qrels = df_qrels[df_qrels['topic_id'] == topic]

    # 5. Loop through each system
    for system in systems:
        # filter system results to extract:
        current_run = df_runs[
            (df_runs['System_Name'] == system) & # only results from this system
            (df_runs['Topic_ID'] == topic) # only for this topic
        ]

        # 6. Loop through all depths
        for k in depths:
            # 7. handle empty results to prevent errors and keep matrix complete
            if current_run.empty: # if a system returned nothing
                topic_results[k][system] = 0  # precision=0
            # compute Precision@K
            else:
                # call the function, take top K results, compare with relevant docs, return precision
                topic_results[k][system] = calculate_precision_at_k(current_run, current_qrels, k)

    # 7. Save results per topic
    for k in depths:
        results[k].append(topic_results[k])

# 8. Convert to matrices
matrices = {
    # convert list to dataframe, set topic as row index
    k: pd.DataFrame(results[k]).set_index('Topic')
    for k in depths # for each depth
}

# 9. Display results
for k in depths:
    print(f"Precision@{k} Matrix Preview:")
    print(matrices[k].head())
    print("\n")


Precision@5 Matrix Preview:
       INQ602  att99ate  plt8ah1  Mer8Adtd4  apl8ctd  orcl99man  CL99XTopt  \
Topic                                                                        
401       0.0       0.4      0.0        0.2      0.0        0.4        0.0   
402       0.8       0.8      0.4        0.6      0.0        0.8        0.6   
403       1.0       0.8      1.0        1.0      0.0        1.0        1.0   
404       0.4       0.6      0.6        0.4      0.0        0.0        0.6   
405       0.6       0.8      0.6        0.4      0.0        0.4        0.6   

       nttd8alx  fub99td  isa50t  ibms99c  mds08a2  acsys8alo2  GE8ATDN2  \
Topic                                                                      
401         0.0      0.6     0.2      1.0      1.0         0.0       0.8   
402         0.6      1.0     0.6      1.0      1.0         0.8       1.0   
403         0.8      1.0     0.8      1.0      1.0         1.0       1.0   
404         0.0      0.6     0.2      0.6    

## 2.4 Export the matrices in csv format

In [13]:
# Save to CSV for the group to use
for k in depths:
    matrices[k].to_csv(f'P@{k}.csv')

print("Phase 1 Complete: Matrices for Precision@5,10,20,50, and 100 generated and saved.")

Phase 1 Complete: Matrices for Precision@5,10,20,50, and 100 generated and saved.
